-------------
We will try running deseq with indepedent factor off, and see whether the filter will look similar to what was ran with it on. 

In [22]:
import scanpy as sc
from pydeseq2.ds import DeseqStats
import Utility.pseudobulk as pseudobulk
from Utility.mark_differential_expressed import generate_differential_expressed_column

In [7]:
adata = sc.read_h5ad("/project/imoskowitz/yubin/1-sc_practice/Data/SmoNull_Brain_system.h5ad")

In [10]:
adata

AnnData object with n_obs × n_vars = 38311 × 33696
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Condition', 'Library.ident', 'Sample', 'Replicate', 'percent.mt', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.1', 'seurat_clusters', 'doublet_finder', 'doublet_status', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'SCT_snn_res.1', 'SCT_snn_res.4', 'Extended_mouse_gastrulation_label', 'System', 'ClusterSystem', 'total_counts', 'size_factors', 'Celltype', 'leiden_0_25_log1p', 'leiden_0_25_scran', 'leiden_0_25_pearson', 'leiden_0_5_log1p', 'leiden_0_5_scran', 'leiden_0_5_pearson', 'leiden_1_log1p', 'leiden_1_scran', 'leiden_1_pearson', 'leiden_2_log1p', 'leiden_2_scran', 'leiden_2_pearson', 'leiden_3_log1p', 'leiden_3_scran', 'leiden_3_pearson', 'leiden_5_log1p', 'leiden_5_scran', 'leiden_5_pearson', 'manual_celltype_annotation'
    var: 'features', 'highly_variable_log1p', 'highly_variable_nbatches_log1p', 'highly_variable_intersection_log1p', 'mean', 'std', 'means', 'variances', 'r

## Pseudobulking

In [11]:
# Convert X to raw
adata.X = adata.layers["raw_counts"]

In [12]:
hindbrain_subset = adata[adata.obs.manual_celltype_annotation == "Hindbrain"]

In [13]:
hindbrain_pb = pseudobulk.run_pseudobulk(hindbrain_subset, by = "Sample", metadatas = "manual_celltype_annotation", min_cells = 0)

/home/yubin/.conda/envs/scRNAseq_python/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:293: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


## Deseq

In [ ]:
hindbrain_dds = pseudobulk.run_deseq(hindbrain_pb, "~Condition", min_samples=0)

Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 2.89 seconds.

Fitting dispersion trend curve...
... done in 0.59 seconds.

Fitting MAP dispersions...
... done in 3.35 seconds.

Fitting LFCs...
... done in 2.32 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.



## Summary Stat - Independent_filter = False

In [16]:

stat_res = DeseqStats(hindbrain_dds, n_cpus=8, contrast=("Condition", "Control", "SmoNull"), independent_filter= False)
stat_res.summary()
de = stat_res.results_df

Running Wald tests...


Log2 fold change & Wald test p-value: Condition Control vs SmoNull
                      baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Xkr4                489.923797       -0.173432  0.145978 -1.188072  0.234805   
Gm1992               23.294824       -0.051467  0.418526 -0.122972  0.902129   
Gm19938              12.049165       -0.394003  0.563621 -0.699056  0.484517   
Gm37381               0.504649       -2.149422  3.358629 -0.639970  0.522192   
Rp1                   7.060131       -0.226565  0.788342 -0.287394  0.773811   
...                        ...             ...       ...       ...       ...   
ENSMUSG00000095523    0.000000             NaN       NaN       NaN       NaN   
ENSMUSG00000095475    0.000000             NaN       NaN       NaN       NaN   
ENSMUSG00000094855    0.000000             NaN       NaN       NaN       NaN   
ENSMUSG00000095019    0.000000             NaN       NaN       NaN       NaN   
ENSMUSG00000095041  299.150421       -0.544666  0.257

... done in 12.49 seconds.



In [19]:
de.tail()

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
ENSMUSG00000095523,0.000000,NaN,NaN,NaN,NaN,NaN
ENSMUSG00000095475,0.000000,NaN,NaN,NaN,NaN,NaN
ENSMUSG00000094855,0.000000,NaN,NaN,NaN,NaN,NaN
ENSMUSG00000095019,0.000000,NaN,NaN,NaN,NaN,NaN
ENSMUSG00000095041,299.150421,-0.544666,0.257583,-2.114525,0.03447,0.182416


## Adding column for is_differntially_expressed

In [23]:
de = generate_differential_expressed_column(de)

In [26]:
de.iloc[:, -1].count()

33696

## Generate some more cell filters to go with this independent_filter = False

In [40]:
de_list = [de]
for i in range(1,7):
    hindbrain_pb = pseudobulk.run_pseudobulk(hindbrain_subset, by = "Sample", metadatas = "manual_celltype_annotation", min_cells = i)
    hindbrain_dds = pseudobulk.run_deseq(hindbrain_pb, "~Condition", min_cells= 0) # should be min_samples, but I am too lazy to reload kernal. 
    stat_res = DeseqStats(hindbrain_dds, n_cpus=8, contrast=("Condition", "Control", "SmoNull"), independent_filter=False)
    stat_res.summary()
    de_list.append(stat_res.results_df)

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 2.85 seconds.

Fitting dispersion trend curve...
... done in 0.75 seconds.

Fitting MAP dispersions...
... done in 3.33 seconds.

Fitting LFCs...
... done in 2.36 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 11.72 seconds.



Log2 fold change & Wald test p-value: Condition Control vs SmoNull
                      baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Xkr4                489.923797       -0.173432  0.145978 -1.188072  0.234805   
Gm1992               23.294824       -0.051467  0.418526 -0.122972  0.902129   
Gm19938              12.049165       -0.394003  0.563621 -0.699056  0.484517   
Gm37381               0.504649       -2.149422  3.358629 -0.639970  0.522192   
Rp1                   7.060131       -0.226565  0.788342 -0.287394  0.773811   
...                        ...             ...       ...       ...       ...   
ENSMUSG00000095500    0.375806        0.649481  3.541146  0.183410  0.854477   
Csprs                 0.132808       -0.312305  4.427542 -0.070537  0.943766   
ENSMUSG00000096808    0.885288       -1.580838  2.209688 -0.715412  0.474354   
ENSMUSG00000095742   10.259770        0.051710  0.588223  0.087909  0.929949   
ENSMUSG00000095041  299.150421       -0.544666  0.257

Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 2.72 seconds.

Fitting dispersion trend curve...
... done in 0.54 seconds.

Fitting MAP dispersions...
... done in 3.13 seconds.

Fitting LFCs...
... done in 2.12 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 11.57 seconds.



Log2 fold change & Wald test p-value: Condition Control vs SmoNull
                      baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Xkr4                489.923797       -0.173436  0.146054 -1.187484  0.235037   
Gm1992               23.294824       -0.051404  0.421162 -0.122052  0.902858   
Gm19938              12.049165       -0.394036  0.567454 -0.694393  0.487436   
Gm37381               0.504649       -2.149261  3.379678 -0.635937  0.524818   
Rp1                   7.060131       -0.226287  0.793209 -0.285280  0.775429   
...                        ...             ...       ...       ...       ...   
ENSMUSG00000079190    0.265616       -1.215036  4.324039 -0.280996  0.778714   
ENSMUSG00000095500    0.375806        0.649481  3.567591  0.182050  0.855543   
ENSMUSG00000096808    0.885288       -1.581067  2.223052 -0.711215  0.476951   
ENSMUSG00000095742   10.259770        0.051623  0.592304  0.087156  0.930547   
ENSMUSG00000095041  299.150421       -0.544661  0.257

...

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 2.86 seconds.

Fitting dispersion trend curve...
... done in 0.51 seconds.

Fitting MAP dispersions...
... done in 2.92 seconds.

Fitting LFCs...
... done in 2.03 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 11.54 seconds.



Log2 fold change & Wald test p-value: Condition Control vs SmoNull
                       baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Xkr4                 489.923797       -0.173413  0.145601 -1.191013  0.233648   
Gm1992                23.294824       -0.051396  0.421504 -0.121934  0.902951   
Gm19938               12.049165       -0.394049  0.569021 -0.692504  0.488621   
Rp1                    7.060131       -0.226432  0.790645 -0.286390  0.774580   
Sox17                 17.177941       -0.554050  0.485131 -1.142064  0.253428   
...                         ...             ...       ...       ...       ...   
mt-Nd6               191.468738       -0.126748  0.485363 -0.261140  0.793985   
mt-Cytb             1630.586234        0.386753  0.165523  2.336548  0.019463   
ENSMUSG00000096808     0.885288       -1.581152  2.228099 -0.709642  0.477926   
ENSMUSG00000095742    10.259770        0.051565  0.595067  0.086655  0.930946   
ENSMUSG00000095041   299.150421       -0.5

Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 2.53 seconds.

Fitting dispersion trend curve...
... done in 0.51 seconds.

Fitting MAP dispersions...
... done in 3.16 seconds.

Fitting LFCs...
... done in 2.00 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: Condition Control vs SmoNull
                       baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Xkr4                 489.923797       -0.173410  0.145529 -1.191581  0.233426   
Gm1992                23.294824       -0.051419  0.420525 -0.122273  0.902683   
Gm19938               12.049165       -0.394039  0.567785 -0.693994  0.487686   
Rp1                    7.060131       -0.226571  0.788233 -0.287441  0.773774   
Sox17                 17.177941       -0.554095  0.484074 -1.144649  0.252354   
...                         ...             ...       ...       ...       ...   
mt-Nd6               191.468738       -0.126748  0.485363 -0.261140  0.793985   
mt-Cytb             1630.586234        0.386753  0.165331  2.339262  0.019322   
ENSMUSG00000096808     0.885288       -1.581078  2.223671 -0.711021  0.477071   
ENSMUSG00000095742    10.259770        0.051589  0.593943  0.086858  0.930784   
ENSMUSG00000095041   299.150421       -0.5

... done in 11.49 seconds.



## Comparing with Venn

In [60]:
from matplotlib_venn import venn3
from Utility.venn_diagram_utils import create_labeled_venn3
import pandas as pd
import numpy as np

In [41]:
control = pd.read_csv("/project/imoskowitz/yubin/1-sc_practice/Differential_Gene_Analysis/DataFrames/Pre-pseudobulk-filter/DEG_Hindbrain_Control_vs_SmoNull_0_cells")

In [49]:
control

,Unnamed: 0.1,Unnamed: 0,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Significance,diff_expr
0,0,Xkr4,489.923797,-0.173432,0.145978,-1.188072,0.234805,0.444052,False,NS
1,1,Gm1992,23.294824,-0.051467,0.418526,-0.122972,0.902129,0.953270,False,NS
2,2,Gm19938,12.049165,-0.394003,0.563621,-0.699056,0.484517,0.690916,False,NS
3,3,Gm37381,0.504649,-2.149422,3.358629,-0.639970,0.522192,NaN,False,NS
4,4,Rp1,7.060131,-0.226565,0.788342,-0.287394,0.773811,0.882547,False,NS
...,...,...,...,...,...,...,...,...,...,...
33691,33691,ENSMUSG00000095523,0.000000,NaN,NaN,NaN,NaN,NaN,False,NS
33692,33692,ENSMUSG00000095475,0.000000,NaN,NaN,NaN,NaN,NaN,False,NS
33693,33693,ENSMUSG00000094855,0.000000,NaN,NaN,NaN,NaN,NaN,False,NS
33694,33694,ENSMUSG00000095019,0.000000,NaN,NaN,NaN,NaN,NaN,False,NS


In [62]:
de_list[0]

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Significance,diff_expr
Xkr4,489.923797,-0.173432,0.145978,-1.188072,0.234805,0.642775,False,NS
Gm1992,23.294824,-0.051467,0.418526,-0.122972,0.902129,0.983821,False,NS
Gm19938,12.049165,-0.394003,0.563621,-0.699056,0.484517,0.908041,False,NS
Gm37381,0.504649,-2.149422,3.358629,-0.639970,0.522192,0.932194,False,NS
Rp1,7.060131,-0.226565,0.788342,-0.287394,0.773811,0.983821,False,NS
...,...,...,...,...,...,...,...,...
ENSMUSG00000095523,0.000000,NaN,NaN,NaN,NaN,NaN,False,NS
ENSMUSG00000095475,0.000000,NaN,NaN,NaN,NaN,NaN,False,NS
ENSMUSG00000094855,0.000000,NaN,NaN,NaN,NaN,NaN,False,NS
ENSMUSG00000095019,0.000000,NaN,NaN,NaN,NaN,NaN,False,NS


In [61]:
de_list_set = [set(de_list[1].iloc[np.array(de_list[1].loc[:,"diff_expr"] == "up"),1]) for i in range(len(de_list))]

KeyError: 'diff_expr'

In [57]:
control[1, control.loc["diff_expr"] == "up"]

KeyError: 'diff_expr'

# extracting the up regulated genes 

In [ ]:
set1 = control[control.loc["diff_expr"] == "up", 1],
set2 = de_list[0][de_list[0].loc["diff_expr"] == "up",1], 
set3 = de_list[1][de_list[1].loc["diff_expr"] == "up",1] # de_list[0] is still in control


KeyError: 'diff_expr'